In [1]:
import os

from langchain_classic import text_splitter
from langchain_classic.agents.agent_toolkits import vectorstore
from langgraph import graph
from langgraph.graph import StateGraph,END,START
from typing import Dict,TypedDict,List,Union,Annotated,Sequence
from langchain_core.messages import HumanMessage, AIMessage,BaseMessage,ToolMessage,SystemMessage
from dotenv import load_dotenv
from langgraph.graph.message import add_messages
from langchain_deepseek import ChatDeepSeek
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode, tool_node
from langchain_chroma import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings
from langchain_core.embeddings import Embeddings



In [2]:
load_dotenv()
llm = ChatDeepSeek(model="deepseek-chat", temperature=0)

pdf_path = r"F:/desktop/SYSU_Topo a 1-arc-minute global bathymetry from SWOT-derived gravity using the gravity-geological method.pdf"
pdf_loader = PyPDFLoader(pdf_path)

try:
    pages = pdf_loader.load()
    print(f'PDF has {len(pages)} pages')
except Exception as e:
    print(f'Error loading PDF: {str(e)}')
    raise

PDF has 14 pages


In [3]:
#非标准openai embedding，做的适配
import requests
from langchain_core.embeddings import Embeddings
from typing import List
class AcedataEmbedding(Embeddings):
    def __init__(self):
        self.token = os.getenv("ACEDATA_TOKEN")
        self.url = os.getenv("ACEDATA_EMBEDDING_URL")
        self.headers = {
            "accept": "application/json",
            "authorization": f"Bearer {self.token}",
            "content-type": "application/json"
        }

    def embed_documents(self, texts: List[str]) -> List[List[float]]:
        payload = {
            "input": texts,
            "model": "text-embedding-3-small",
            "encoding_format": "float"
        }
        resp = requests.post(self.url, json=payload, headers=self.headers)
        resp.raise_for_status()
        data = resp.json()
        return [item["embedding"] for item in data["data"]]

    def embed_query(self, text: str) -> List[float]:
        return self.embed_documents([text])[0]

# 实例化，直接给Chroma用
embeddings = AcedataEmbedding()

In [4]:
# text_splitter = RecursiveCharacterTextSplitter(chunk_size=3000,chunk_overlap =500)
#
# pages_split = text_splitter.split_documents(pages)
# # 存向量化结果
# persist_directory = r"F:\desktop\langgraph\embedding"
# collection_name = "literature"
#
# try:
#     vectorstore = Chroma.from_documents(
#         documents=pages_split,
#         embedding=embeddings,
#         persist_directory=persist_directory,
#         collection_name=collection_name,
#     )
#     print(f'created vectorstore !')
# except Exception as e:
#     print(f'Error created vectorstore: {str(e)}')
#     raise
#
# retriever = vectorstore.as_retriever(
#     search_type = 'similarity',
#     search_kwargs = {'k':5}
# )

In [5]:
@tool
def retriever_tool(query: str) -> str:
    """This tool searches and returns the information from the PDF"""
    docs = retriever.invoke(query)
    if not docs:
        return "I found no relevant information in the PDF."

    results = []
    for i, doc in enumerate(docs):
        results.append(f"Document {i+1}:\n{doc.page_content}")
    return "\n\n".join(results)

tools = [retriever_tool]
llm = llm.bind_tools(tools)

class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], add_messages]

def should_continue(state:AgentState) :
    "check if message contains tool calls"

    result = state["messages"][-1]
    return hasattr(result,'tool_calls') and len(result.tool_calls)>0

system_prompt = """
You are a literature research assistant.
You MUST only use the content from the retrieved document fragments to answer questions.
Do NOT fabricate facts, numbers or opinions that do NOT appear in the provided context.

When you answer:
1. First give your answer based strictly on the context.
2. Then list the source fragments you used, mark each fragment with its Document number.
3. If no relevant content is found in retrieved documents, clearly state: "No relevant literature fragment found." Do not guess.

Context:
{context}

User question:
{question}"""

In [6]:
tools_dict = {our_tools.name: our_tools for our_tools in tools}

def call_llm(state:AgentState) ->AgentState:
    "function to call the LLM with the current state"
    messages = list(state['messages'])
    messages = [SystemMessage(content = system_prompt)] + messages
    messages = llm.invoke(messages)
    return {"messages": [messages]}

In [7]:
def take_action(state: AgentState) -> AgentState:
    """Execute tool calls from the LLM's response."""
    tool_calls = state['messages'][-1].tool_calls
    results = []
    for t in tool_calls:
        print(f"Calling Tool: {t['name']} with query: {t['args'].get('query', 'No query provided')}")

        if not t['name'] in tools_dict: # Checks if a valid tool is present
            print(f"\nTool: {t['name']} does not exist.")
            result = "Incorrect Tool Name, Please Retry and Select tool from List of Available tools."
        else:
            result = tools_dict[t['name']].invoke(t['args'].get('query', ''))
            print(f"Result length: {len(str(result))}")

        # Appends the Tool Message
        results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))

    print("Tools Execution Complete. Back to the model!")
    return {'messages': results}


In [8]:
graph = StateGraph(AgentState)

graph.add_node("llm",call_llm)
graph.add_node("retriever_agent",take_action)

graph.set_entry_point("llm")
graph.add_edge("retriever_agent","llm")

graph.add_conditional_edges("llm",should_continue,{True:"retriever_agent",False:END})

rag_agent= graph.compile()

In [9]:
def running_agent():
    print("\n=== RAG AGENT===")

    while True:
        user_input = input("\nWhat is your question: ")
        if user_input.lower() in ['exit', 'quit']:
            break

        messages = [HumanMessage(content=user_input)] # converts back to a HumanMessage type
        result = rag_agent.invoke({"messages": messages})

        print("\n=== ANSWER ===")
        print(result['messages'][-1].content)

running_agent()


=== RAG AGENT===

=== ANSWER ===
I can only answer questions based on the retrieved document fragments, and your question ("how is the weather today") is not something I can address from literature content.

No relevant literature fragment found.


KeyboardInterrupt: Interrupted by user